# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Хвостова Олександра'
ГРУПА        = 'ШІ-32'
НОМЕР        = 12

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 12   ·   Хвостова Олександра   ·   ШІ-32

Ваші вісім стовпців:
   1. Телефон
   2. Примітка
   3. ступінь КТ
   4. Виписка
   5. spO2 випис
   6. АТ стац
   7. Д-дим2
   8. Гемогл2

Ваша пара для аналізу зв’язку: ЛН та помер/жив

Ваше додаткове питання:
   Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Примітка,ступінь КТ,Виписка,spO2 випис,АТ стац,Д-дим2,Гемогл2
0,0958469788 дружина Ганна,NaN,сердній,2022-03-10 00:00:00,NaN,150/100,NaN,144
1,"0671778486, 0994887663 дружина Оксана","отримував актемра, часті носові кровотечі, втрата нюху",середній,2022-03-16 00:00:00,95,130/80,744,137
2,"0958768045, чоловік Степан 0951784726",часта втрата свідомосі в стаціонарі,критичний,2022-01-18 00:00:00,93,140/90,NaN,105
3,0959819331. дочка Валентина 0960845205,панкреонекроз,критичний,2022-01-13 00:00:00,93,150/100,NaN,116
4,"0952294866, дочка Дарина 0678360161",NaN,NaN,2022-03-11 00:00:00,93,130/90,NaN,124
5,0959922983 сестра Софія,NaN,середній,2021-12-26 00:00:00,NaN,170/100,NaN,120
6,"0998073702, дружина Наталія 0998382226","глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,2022-05-11 00:00:00,96,130/80,NaN,151
7,"0637258771, син Сергій 0968413301",NaN,легкий,2022-05-28 00:00:00,94,120/80,1370,137


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Телефон     120 non-null    object
 1   Примітка    45 non-null     str   
 2   ступінь КТ  42 non-null     str   
 3   Виписка     120 non-null    object
 4   spO2 випис  68 non-null     object
 5   АТ стац     120 non-null    str   
 6   Д-дим2      30 non-null     object
 7   Гемогл2     105 non-null    object
dtypes: object(5), str(3)
memory usage: 13.5+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- spO2 випис
    непорожніх 68 з 121 | унікальних 11 | типи: int×66, str×2
    прик

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'Телефон':    ('ідентифікатор', 'Номери телефонів родичів із приміткою про родинний зв’язок лише ідентифікують контакт і не є величиною.'),
    'Примітка':   ('номінальна', 'Вільний текст лікаря: значення можна лише порівняти на рівність, порядку й відстаней між ними немає.'),
    'ступінь КТ': ('порядкова', 'Ступені від легкого до критичного мають природний порядок, але відстані між сусідніми ступенями невідомі.'),
    'Виписка':    ('інтервальна', 'Календарна дата: різниця між датами має сенс, а нуль відліку умовний.'),
    'spO2 випис': ('відношень', 'Сатурація у відсотках має природний нуль, тому відношення значень мають сенс.'),
    'АТ стац':    ('відношень', 'Систолічний і діастолічний тиск у мм рт. ст. мають природний нуль; у клітинці записано два значення через «/».'),
    'Д-дим2':     ('відношень', 'Концентрація Д-димеру має природний нуль, тому відношення значень мають сенс.'),
    'Гемогл2':    ('відношень', 'Концентрація гемоглобіну в г/л має природний нуль, тому відношення значень мають сенс.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
pd.set_option('display.max_colwidth', None)
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,Номери телефонів родичів із приміткою про родинний зв’язок лише ідентифікують контакт і не є величиною.
1,Примітка,номінальна,"Вільний текст лікаря: значення можна лише порівняти на рівність, порядку й відстаней між ними немає."
2,ступінь КТ,порядкова,"Ступені від легкого до критичного мають природний порядок, але відстані між сусідніми ступенями невідомі."
3,Виписка,інтервальна,"Календарна дата: різниця між датами має сенс, а нуль відліку умовний."
4,spO2 випис,відношень,"Сатурація у відсотках має природний нуль, тому відношення значень мають сенс."
5,АТ стац,відношень,Систолічний і діастолічний тиск у мм рт. ст. мають природний нуль; у клітинці записано два значення через «/».
6,Д-дим2,відношень,"Концентрація Д-димеру має природний нуль, тому відношення значень мають сенс."
7,Гемогл2,відношень,"Концентрація гемоглобіну в г/л має природний нуль, тому відношення значень мають сенс."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']

def очистити_кт(х):
    х = str(х).strip().lower()
    if х.startswith('легк'):
        return 'легкий'
    if х.startswith('серед') or х.startswith('серд'):
        return 'середній'
    return х

def числа(стовпець):
    ч = pd.to_numeric(df[стовпець], errors='coerce').dropna()
    return ч[ч > 0]

def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    if ш == 'номінальна':
        р['мода'] = з.mode().iloc[0]
        р['частоти'] = з.value_counts().head(3).to_dict()
        return р

    if ш == 'порядкова':
        з = з.map(очистити_кт)
        р['унікальних після очищення'] = з.nunique()
        р['мода'] = з.mode().iloc[0]
        р['частоти'] = з.value_counts().to_dict()
        к = pd.Categorical(з, categories=ПОРЯДОК_КТ, ordered=True).codes
        к = pd.Series(к)
        р['медіана'] = ПОРЯДОК_КТ[int(к.median())]
        р['квантилі 25/75'] = (ПОРЯДОК_КТ[int(к.quantile(.25))], ПОРЯДОК_КТ[int(к.quantile(.75))])
        return р

    if ш == 'інтервальна':
        д = pd.to_datetime(df[стовпець], errors='coerce').dropna()
        р['мода'] = д.mode().iloc[0].date()
        р['медіана'] = д.median().date()
        р['середнє'] = д.mean().date()
        р['ст. відхилення, днів'] = round(д.std().days + д.std().seconds / 86400, 1)
        return р

    if стовпець == 'АТ стац':
        ат = df[стовпець].dropna().astype(str).str.strip().str.split('/', expand=True).astype(float)
        ат.columns = ['систолічний', 'діастолічний']
        for к in ат:
            р[к] = {'мода': ат[к].mode().iloc[0], 'медіана': ат[к].median(),
                    'середнє': round(ат[к].mean(), 1), 'ст. відхилення': round(ат[к].std(), 1),
                    'коеф. варіації': round(ат[к].std() / ат[к].mean(), 3)}
        return р

    ч = числа(стовпець)
    р['мода'] = ч.mode().iloc[0]
    р['частоти'] = ч.value_counts().head(3).to_dict()
    р['медіана'] = ч.median()
    р['квантилі 25/75'] = (ч.quantile(.25), ч.quantile(.75))
    р['середнє'] = round(ч.mean(), 2)
    р['ст. відхилення'] = round(ч.std(), 2)
    р['коеф. варіації'] = round(ч.std() / ч.mean(), 3)
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Примітка  (номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': 'Поліалергетик', 'частоти': {'Поліалергетик': 2, 'отримував актемра, часті носові кровотечі, втрата нюху': 1, 'часта втрата свідомосі в стаціонарі': 1}}

--- ступінь КТ  (порядкова)
    {'непорожніх': 42, 'унікальних': 12, 'унікальних після очищення': 5, 'мода': 'середній', 'частоти': {'середній': 18, 'важкий': 10, 'легкий': 7, 'вкрай важкий': 4, 'критичний': 3}, 'медіана': 'середній', 'квантилі 25/75': ('середній', 'важкий')}

--- Виписка  (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'мода': datetime.date(2021, 9, 7), 'медіана': datetime.date(2022, 3, 15), 'середнє': datetime.date(2022, 1, 29), 'ст. відхилення, днів': 109.5}

--- spO2 випис  (відношень)
    {'непорожніх': 68, 'унікальних': 11, 'мода': np.float64(97.0), 'частоти': {97.0: 12, 96.0: 11, 98.0: 9}, 'медіана': np.float64(96.0), 'квантилі 25/75': (np.float64(94.0), np

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq
Телефон,120,120,0958469788 дружина Ганна,1
Примітка,45,44,Поліалергетик,2
ступінь КТ,42,12,середній,10
Виписка,120,90,2021-11-19 00:00:00,3
spO2 випис,68,11,97,12
АТ стац,120,21,130/80,63
Д-дим2,30,30,744,1
Гемогл2,105,60,144,5


In [10]:
ТЕЛ = pd.to_numeric(df['Телефон'], errors='coerce').dropna()
КОДИ_КТ = df['ступінь КТ'].dropna().map(очистити_кт).map({н: i for i, н in enumerate(ПОРЯДОК_КТ)})
ДАТИ = pd.to_datetime(df['Виписка'], errors='coerce').dropna()
ДНІ = (ДАТИ - pd.Timestamp('1970-01-01')).dt.days

БЕЗГЛУЗДІ = [
    ('Телефон', 'середнє', round(ТЕЛ.mean()),
     'Телефон — ідентифікатор: середнє номерів телефонів не відповідає жодній величині.'),
    ('ступінь КТ', 'середнє кодів', round(КОДИ_КТ.mean(), 2),
     'Ступінь — порядкова шкала: коди 0–4 задають лише порядок, відстані між ступенями невідомі, середнє не має змісту.'),
    ('Виписка', 'коефіцієнт варіації', round(ДНІ.std() / ДНІ.mean(), 4),
     'Дата — інтервальна шкала: нуль відліку умовний (1970 р.), тож відношення std/mean залежить від вибору початку й не має змісту.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє,7.587990e+08,Телефон — ідентифікатор: середнє номерів телефонів не відповідає жодній величині.
1,ступінь КТ,середнє кодів,1.480000e+00,"Ступінь — порядкова шкала: коди 0–4 задають лише порядок, відстані між ступенями невідомі, середнє не має змісту."
2,Виписка,коефіцієнт варіації,5.800000e-03,"Дата — інтервальна шкала: нуль відліку умовний (1970 р.), тож відношення std/mean залежить від вибору початку й не має змісту."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: ЛН × помер/жив

ЛН: шкала — не у вашому списку
помер/жив: шкала — не у вашому списку

       ЛН  помер/жив
0  ІІІ-IV          1
1  ІІ-ІІІ          2
2      ІІ          2
3  ІІІ-IV          2
4  ІІ-ІІІ          2
5  ІІ-ІІІ          1


In [12]:
from scipy.stats import chi2_contingency

ОБРАНА_МІРА = 'Коефіцієнт Крамера (V) та χ²'
ЧОМУ_САМЕ_ВОНА = 'ЛН — порядкова шкала, помер/жив — бінарна номінальна; для пари номінальна × порядкова обирають коефіцієнт Крамера (V).'

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

таблиця = pd.crosstab(df[A], df[B])
print(таблиця)
print()
хі2, p, ст_вільн, очік = chi2_contingency(таблиця)
n = таблиця.values.sum()
V = np.sqrt(хі2 / (n * (min(таблиця.shape) - 1)))
print(f'χ² = {хі2:.2f}, ст. вільності = {ст_вільн}, p = {p:.2e}')
print(f'V Крамера = {V:.3f}')
print(f'Клітинок із очікуваною частотою < 5: {(очік < 5).sum()} з {очік.size}')
print()
print('Зв’язок сильний: V = %.2f. Усі пацієнти зі ступенем І, І-ІІ та ІІ вижили,' % V)
print('а за ступеня ІІ-ІІІ померли 49 із 59.')

помер/жив   1   2
ЛН               
І           0  12
І-ІІ        0   9
ІІ          0  21
ІІ-ІІІ     49  10
ІІІ        11   6
ІІІ-IV      1   2

χ² = 69.58, ст. вільності = 5, p = 1.25e-13
V Крамера = 0.758
Клітинок із очікуваною частотою < 5: 4 з 12

Зв’язок сильний: V = 0.76. Усі пацієнти зі ступенем І, І-ІІ та ІІ вижили,
а за ступеня ІІ-ІІІ померли 49 із 59.


In [13]:
alf = df[A].astype('category').cat.codes
інший = df[A].map({'ІІІ': 0, 'І': 1, 'ІІ-ІІІ': 2, 'ІІІ-IV': 3, 'І-ІІ': 4, 'ІІ': 5})

print('Пірсон між ЛН і помер/жив:')
print('  коди за алфавітом   :', round(alf.corr(df[B]), 3))
print('  коди в іншому порядку:', round(інший.corr(df[B]), 3))
print()
print('Знак і величина r змінюються разом із довільним порядком кодів ЛН,')
print('а помер/жив — номінальна змінна. Пірсон припускає числові шкали')
print('з рівними інтервалами, тому отримане число інтерпретувати не можна.')

Пірсон між ЛН і помер/жив:
  коди за алфавітом   : -0.544
  коди в іншому порядку: 0.424

Знак і величина r змінюються разом із довільним порядком кодів ЛН,
а помер/жив — номінальна змінна. Пірсон припускає числові шкали
з рівними інтервалами, тому отримане число інтерпретувати не можна.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'Телефон':    'Ідентифікатор: у модель не беремо взагалі.',
    'Примітка':   'Вільний текст: у модель не беремо; можливі окремі бінарні ознаки за ключовими словами.',
    'ступінь КТ': 'Очищення написань і впорядкована категорія легкий < середній < важкий < вкрай важкий < критичний.',
    'Виписка':    'Дата: перехід до тривалості перебування в днях (Виписка − Поступлення), далі стандартизація.',
    'spO2 випис': 'Нулі замінити пропусками, далі стандартизація або нормалізація.',
    'АТ стац':    'Розбити на систолічний і діастолічний тиск, далі стандартизація кожного.',
    'Д-дим2':     'Привести значення до однієї одиниці, далі логарифмування й стандартизація.',
    'Гемогл2':    'Текст «немає» замінити пропуском, далі стандартизація.',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,Ідентифікатор: у модель не беремо взагалі.
1,Примітка,номінальна,Вільний текст: у модель не беремо; можливі окремі бінарні ознаки за ключовими словами.
2,ступінь КТ,порядкова,Очищення написань і впорядкована категорія легкий < середній < важкий < вкрай важкий < критичний.
3,Виписка,інтервальна,"Дата: перехід до тривалості перебування в днях (Виписка − Поступлення), далі стандартизація."
4,spO2 випис,відношень,"Нулі замінити пропусками, далі стандартизація або нормалізація."
5,АТ стац,відношень,"Розбити на систолічний і діастолічний тиск, далі стандартизація кожного."
6,Д-дим2,відношень,"Привести значення до однієї одиниці, далі логарифмування й стандартизація."
7,Гемогл2,відношень,"Текст «немає» замінити пропуском, далі стандартизація."


In [15]:
# порядкова: ступінь КТ
кт_порядок = pd.Categorical(df['ступінь КТ'].dropna().map(очистити_кт),
                            categories=ПОРЯДОК_КТ, ordered=True)
print('ступінь КТ:', pd.Series(кт_порядок.codes).value_counts().sort_index().to_dict())

# інтервальна: тривалість перебування замість дати
тривалість = (pd.to_datetime(df['Виписка'], errors='coerce')
              - pd.to_datetime(df['Поступлення'], errors='coerce')).dt.days
тривалість = тривалість.where(тривалість >= 0)
тривалість_z = (тривалість - тривалість.mean()) / тривалість.std()
print('Тривалість, днів: середнє %.1f, std %.1f' % (тривалість.mean(), тривалість.std()))
print('Стандартизована: середнє %.2f, std %.2f' % (тривалість_z.mean(), тривалість_z.std()))

# відношень: гемоглобін
гемогл = pd.to_numeric(df['Гемогл2'], errors='coerce')
assert гемогл.nunique() > 1
гемогл_z = (гемогл - гемогл.mean()) / гемогл.std()
print('Гемогл2 стандартизований: середнє %.2f, std %.2f' % (гемогл_z.mean(), гемогл_z.std()))

ступінь КТ: {0: 7, 1: 18, 2: 10, 3: 4, 4: 3}
Тривалість, днів: середнє 19.6, std 20.9
Стандартизована: середнє -0.00, std 1.00
Гемогл2 стандартизований: середнє 0.00, std 1.00


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    (1, 'Номінальна шкала: помер/жив закодовано 1 і 2, і від нього обчислено середнє.',
        'Коди 1 і 2 — лише мітки категорій, тому середнє 1.50 не означає «посередині між результатами» і не має змісту.',
        'Подати частоти: 61 помер (50.4%) і 60 вижили (49.6%); мода — «помер».'),
    (2, 'Дані не придатні: одна ознака записана різними написаннями, тому «статей» нараховано чотири.',
        'Значення «ж» і «жін», «ч» і «чол» позначають одне й те саме, тож статей дві, а не чотири.',
        'Об’єднати написання й порахувати частоти: ж — 75 (62%), ч — 46 (38%).'),
    (3, 'Порядкова шкала: від ступеня обчислено середнє кодів; категорії ще й записані по-різному.',
        'Коди задають лише порядок, відстаней між ступенями немає; сортування за алфавітом ламає порядок, а варіанти написання дублюють категорії.',
        'Очистити назви, задати порядок легкий < … < критичний; обчислити медіану й моду (мода та медіана — «середній»).'),
    (4, 'Дані не придатні: у КТ частина значень записана текстом-діапазоном («20-30%», «до 17%»).',
        'Через errors=\'coerce\' діапазони стали пропусками, середнє пораховано лише за 27 з 44 заповнених значень і не відображає вибірку.',
        'Перетворити діапазони на числа (наприклад, середина діапазону) або вказати, що показник визначений лише для 27 пацієнтів.'),
    (5, 'Дані не придатні: у spO2 є значення 0.9, імовірно, ввели не у відсотках; воно спотворює мінімум і середнє.',
        'Мінімум 0.9% фізіологічно неможливий для живого пацієнта, він знижує середнє 82.73 до 82.05; висновок про «виражену недостатність» спирається на помилковий запис.',
        'Виправити або виключити значення 0.9, повідомляти медіану (84) разом із розкидом.'),
    (6, 'Інтервальна шкала: коефіцієнт варіації обчислено для дат.',
        'Нуль дат умовний (1970 р.), тому CV залежить від початку відліку; значення менше 1% говорить лише про великі числа днів від епохи.',
        'Описувати потік розмахом і стандартним відхиленням у днях (розмах 463 дні) або кількістю госпіталізацій за місяць.'),
    (7, 'Номінальна шкала: кореляцію Пірсона застосовано до двох закодованих номінальних змінних.',
        'Коди 0/1 для статі не мають числового змісту; Пірсон показує лише лінійну залежність між числами, а «r близьке до нуля» хибне: r = 0.18, p = 0.052.',
        'Таблиця спряженості, χ² і коефіцієнт Крамера V (V = 0.18, p ≈ 0.05); висновок «зв’язку немає» не обґрунтований.'),
    (8, 'Номінальні та сталі стовпці включено до min-max нормалізації.',
        'Нормалізовано також помер/жив (категоріальний код), а ФЕРИТ2 має одне значення, тому max − min = 0 і ділення на нуль дає NaN.',
        'Нормалізувати лише змістовно числові стовпці з nunique > 1; категоріальні кодувати окремо, сталі виключити.'),
    (9, 'Дані не придатні: dropna() видаляє всі рядки, бо в кожному рядку є хоча б один пропуск.',
        'Після видалення залишилось 0 рядків із 121, тож «повністю заповнений набір» порожній; пропуски часто структурні (дата розтину є лише в померлих).',
        'Видалити стовпці з надмірною кількістю пропусків і обрати відповідну імпутацію, пропуски в Дата розтину не імпутувати.'),
    (10, 'Витік цільової змінної: дата розтину існує лише для померлих.',
         'r = 1.00 означає, що ознака відома лише після результату; крім того, NaT перетворено на велике від’ємне число і кореляцію обчислено на неіснуючих значеннях.',
         'Виключити «Дата розтину» з предикторів і ранжувати лише ознаки, відомі до результату (наприклад, Лім2, ІЛ6-2, Вік).'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"Номінальна шкала: помер/жив закодовано 1 і 2, і від нього обчислено середнє.","Коди 1 і 2 — лише мітки категорій, тому середнє 1.50 не означає «посередині між результатами» і не має змісту.",Подати частоти: 61 помер (50.4%) і 60 вижили (49.6%); мода — «помер».
1,2,"Дані не придатні: одна ознака записана різними написаннями, тому «статей» нараховано чотири.","Значення «ж» і «жін», «ч» і «чол» позначають одне й те саме, тож статей дві, а не чотири.","Об’єднати написання й порахувати частоти: ж — 75 (62%), ч — 46 (38%)."
2,3,Порядкова шкала: від ступеня обчислено середнє кодів; категорії ще й записані по-різному.,"Коди задають лише порядок, відстаней між ступенями немає; сортування за алфавітом ламає порядок, а варіанти написання дублюють категорії.","Очистити назви, задати порядок легкий < … < критичний; обчислити медіану й моду (мода та медіана — «середній»)."
3,4,"Дані не придатні: у КТ частина значень записана текстом-діапазоном («20-30%», «до 17%»).","Через errors='coerce' діапазони стали пропусками, середнє пораховано лише за 27 з 44 заповнених значень і не відображає вибірку.","Перетворити діапазони на числа (наприклад, середина діапазону) або вказати, що показник визначений лише для 27 пацієнтів."
4,5,"Дані не придатні: у spO2 є значення 0.9, імовірно, ввели не у відсотках; воно спотворює мінімум і середнє.","Мінімум 0.9% фізіологічно неможливий для живого пацієнта, він знижує середнє 82.73 до 82.05; висновок про «виражену недостатність» спирається на помилковий запис.","Виправити або виключити значення 0.9, повідомляти медіану (84) разом із розкидом."
5,6,Інтервальна шкала: коефіцієнт варіації обчислено для дат.,"Нуль дат умовний (1970 р.), тому CV залежить від початку відліку; значення менше 1% говорить лише про великі числа днів від епохи.",Описувати потік розмахом і стандартним відхиленням у днях (розмах 463 дні) або кількістю госпіталізацій за місяць.
6,7,Номінальна шкала: кореляцію Пірсона застосовано до двох закодованих номінальних змінних.,"Коди 0/1 для статі не мають числового змісту; Пірсон показує лише лінійну залежність між числами, а «r близьке до нуля» хибне: r = 0.18, p = 0.052.","Таблиця спряженості, χ² і коефіцієнт Крамера V (V = 0.18, p ≈ 0.05); висновок «зв’язку немає» не обґрунтований."
7,8,Номінальні та сталі стовпці включено до min-max нормалізації.,"Нормалізовано також помер/жив (категоріальний код), а ФЕРИТ2 має одне значення, тому max − min = 0 і ділення на нуль дає NaN.","Нормалізувати лише змістовно числові стовпці з nunique > 1; категоріальні кодувати окремо, сталі виключити."
8,9,"Дані не придатні: dropna() видаляє всі рядки, бо в кожному рядку є хоча б один пропуск.","Після видалення залишилось 0 рядків із 121, тож «повністю заповнений набір» порожній; пропуски часто структурні (дата розтину є лише в померлих).","Видалити стовпці з надмірною кількістю пропусків і обрати відповідну імпутацію, пропуски в Дата розтину не імпутувати."
9,10,Витік цільової змінної: дата розтину існує лише для померлих.,"r = 1.00 означає, що ознака відома лише після результату; крім того, NaT перетворено на велике від’ємне число і кореляцію обчислено на неіснуючих значеннях.","Виключити «Дата розтину» з предикторів і ранжувати лише ознаки, відомі до результату (наприклад, Лім2, ІЛ6-2, Вік)."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
# розділ 3: ступінь КТ
сирі = df['ступінь КТ'].dropna()
чисті = pd.Categorical(сирі.map(очистити_кт), categories=ПОРЯДОК_КТ, ordered=True)
коди = pd.Series(чисті.codes)
print('Розділ 3. Неправильно: середній код =', round(коди.mean(), 2))
print('Розділ 3. Правильно: мода =', pd.Series(чисті).mode()[0], '| медіана =', ПОРЯДОК_КТ[int(коди.median())])
print(pd.Series(чисті).value_counts().sort_index().to_string())
print()

# розділ 6: дати
дати = pd.to_datetime(df['Поступлення'], errors='coerce').dropna()
print('Розділ 6. Неправильно: CV дат =', round(100 * (дати.astype('int64') / 10**9 / 86400).std() / (дати.astype('int64') / 10**9 / 86400).mean(), 3), '%')
print('Розділ 6. Правильно: розмах =', (дати.max() - дати.min()).days, 'днів;',
      'std =', round(дати.std().days + дати.std().seconds / 86400, 1), 'днів')
print()

# розділ 9: пропуски
print('Розділ 9. Неправильно: рядків після dropna() =', len(df.dropna()))
пропуски = df.isna().mean().sort_values(ascending=False)
print('Розділ 9. Правильно: стовпців з >50% пропусків:', int((пропуски > .5).sum()), 'з', df.shape[1])
залишок = df.loc[:, пропуски <= .5].dropna(how='any', subset=['Вік', 'помер/жив'])
print('Після видалення таких стовпців рядків із заповненими Вік і помер/жив:', len(залишок))

Розділ 3. Неправильно: середній код = 1.48
Розділ 3. Правильно: мода = середній | медіана = середній
легкий           7
середній        18
важкий          10
вкрай важкий     4
критичний        3

Розділ 6. Неправильно: CV дат = 0.595 %
Розділ 6. Правильно: розмах = 463 днів; std = 113.0 днів

Розділ 9. Неправильно: рядків після dropna() = 0
Розділ 9. Правильно: стовпців з >50% пропусків: 11 з 77
Після видалення таких стовпців рядків із заповненими Вік і помер/жив: 121


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'Тяжкість легеневої недостатності (ЛН) сильно пов’язана з результатом лікування: V Крамера становить 0.76, а всі 42 пацієнти зі ступенями І, І-ІІ та ІІ вижили.',
    'У моєму наборі Телефон є ідентифікатором, ступінь КТ порядковою шкалою, дата виписки інтервальною, а тиск, Д-димер, гемоглобін і сатурація — шкалою відношень, тож середнє коректне лише для останніх чотирьох і для дат.',
    'Багато стовпців потребують очищення перед аналізом: у ступені КТ дванадцять написань п’яти категорій, у spO2 при виписці є 10 нулів, а в Д-димері змішані одиниці вимірювання.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Тяжкість легеневої недостатності (ЛН) сильно пов’язана з результатом лікування: V Крамера становить 0.76, а всі 42 пацієнти зі ступенями І, І-ІІ та ІІ вижили.
2. У моєму наборі Телефон є ідентифікатором, ступінь КТ порядковою шкалою, дата виписки інтервальною, а тиск, Д-димер, гемоглобін і сатурація — шкалою відношень, тож середнє коректне лише для останніх чотирьох і для дат.
3. Багато стовпців потребують очищення перед аналізом: у ступені КТ дванадцять написань п’яти категорій, у spO2 при виписці є 10 нулів, а в Д-димері змішані одиниці вимірювання.


In [21]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'Значення «АТ пост» — рядок, у якому записано два вимірювання: систолічний і діастолічний тиск. '
    'Перший спосіб: розбити рядок на два числові стовпці, САТ і ДАТ. '
    'Другий спосіб: обчислити похідні величини — пульсовий тиск (САТ − ДАТ) або середній артеріальний тиск (ДАТ + (САТ − ДАТ)/3). '
    'Третій спосіб: віднести пацієнта до категорії за порогами тиску (норма, гіпертензія I, II, III ступеня), тобто перетворити на порядкову ознаку. '
    'У модель я візьму перший спосіб: САТ і ДАТ зберігають усю початкову інформацію, лишаються у шкалі відношень (мм рт. ст.), '
    'не вводять довільних порогів, як категорії, а похідні величини при потребі можна отримати з них самих.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.

Значення «АТ пост» — рядок, у якому записано два вимірювання: систолічний і діастолічний тиск. Перший спосіб: розбити рядок на два числові стовпці, САТ і ДАТ. Другий спосіб: обчислити похідні величини — пульсовий тиск (САТ − ДАТ) або середній артеріальний тиск (ДАТ + (САТ − ДАТ)/3). Третій спосіб: віднести пацієнта до категорії за порогами тиску (норма, гіпертензія I, II, III ступеня), тобто перетворити на порядкову ознаку. У модель я візьму перший спосіб: САТ і ДАТ зберігають усю початкову інформацію, лишаються у шкалі відношень (мм рт. ст.), не вводять довільних порогів, як категорії, а похідні величини при потребі можна отримати з них самих.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, як є',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Я запустила ноутбук згори донизу без помилок. Перевірила шкали стовпців за реальними значеннями, кількості рядків і пропусків, V Крамера та χ² для пари ЛН × помер/жив, а також помилки в чужому звіті (0 рядків після dropna, значення 0.9 у spO2, два написання статей, витік дати розтину).'
ПІДТВЕРДЖУЮ = True      # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 12 готова до захисту.
Хвостова Олександра, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.